# L4a: Graph and Tree Representations

A graph represents objects and their connections. We call the objects *vertices* and the connections *edges*. Roads between cities, transformations between chemical species, and calls between functions can all be represented as graphs. 

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe graph structure and measures:__ Define vertices, edges, walks, paths, cycles, and connectivity, including weak and strong connectivity in directed graphs. Calculate vertex degrees from the edges and graph density from vertex and edge counts.
> * __Characterize complete graphs, bipartite graphs, and trees:__ Explain each family's defining properties and use edge counts, coloring, and path properties to determine whether a graph belongs to it.
> * __Compare graph representations:__ Construct adjacency lists and adjacency matrices from an edge list. Explain their storage and access costs, and choose a representation based on graph density and the operations a computation requires.

In this lecture, we examine three important graph families: complete graphs, bipartite graphs, and trees. We then compare edge lists, adjacency matrices, and adjacency lists. The choice of representation determines the memory required to store a graph and the work needed to find an edge or visit a vertex's neighbors.

Let's get started!

___


## Setup, Data, and Prerequisites

First, we load the local [`Include.jl`](Include.jl) setup file and its required packages.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates [`Include.jl`](Include.jl) in the notebook's global scope. The setup file activates the course project, defines local paths, and loads the course package and the `Test` standard library used to check our calculations.

Let's set up our code environment:


In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

See the [Julia documentation](https://docs.julialang.org/en/v1/) for language details and the [`Test` documentation](https://docs.julialang.org/en/v1/stdlib/Test/) for the checks used here. The graph functions are defined in [`GraphRepresentation.jl`](../../../code/src/GraphRepresentation.jl).

Our worked example uses [`data/SimpleGraph.txt`](data/SimpleGraph.txt), which contains seven weighted, directed edges connecting six vertices. We use this graph to construct and compare an adjacency list and an adjacency matrix. The dataset is described in [`data/README.md`](data/README.md).

___


## Simple Graphs

A graph describes which vertices are joined by edges. Before we compare ways to store these connections, let's establish the notation and explain how we can move through a graph.

> __Simple graph:__
>
> A simple graph $\mathcal{G}=(\mathcal{V},\mathcal{E})$ consists of a vertex set $\mathcal{V}$ and an edge set $\mathcal{E}$. The word *simple* rules out self-loops, which connect a vertex to itself, and repeated parallel edges.
>
> * In an __undirected graph__, an edge is an unordered pair $\{u,v\}$ of distinct vertices and can be followed in either direction.
> * In a __directed graph__, an edge is an ordered pair $(u,v)$ with $u\neq v$. It points from $u$ to $v$; the reverse edge $(v,u)$ is a different edge and may not exist.

For example, in a social network, person A may follow person B without B following A. Edges may also carry a __weight__, such as distance, capacity, or cost, whether the graph is directed or undirected.

__How do we describe movement through a graph?__ A __walk__ is a sequence of vertices $v_0,v_1,\ldots,v_k$ in which each consecutive pair is joined by an edge. In a directed graph, the walk must follow the edge directions. A __path__ is a walk with no repeated vertices. A __cycle__ is a closed walk containing at least one edge that repeats no edge and no vertex except its starting vertex, $v_0=v_k$.

These definitions let us describe __connectivity__:

* An undirected graph is __connected__ when a path joins every pair of vertices.
* A directed graph is __weakly connected__ when it becomes connected after edge directions are ignored.
* A directed graph is __strongly connected__ when every ordered pair $(u,v)$ has a directed path from $u$ to $v$.

<div>
    <center>
        <img src="figs/Fig-General-Graph-Schematic.svg" width="980" alt="Left: an undirected graph with six vertices and seven weighted edges, with the degree of vertex 4 marked. Right: the same vertices and edges drawn as a directed acyclic graph, with the in-degree of vertex 2 and the out-degree of vertex 4 marked."/>
    </center>
</div>

The undirected graph is connected; its directed counterpart is weakly but not strongly connected. We can reach vertex 5 from vertex 0, but cannot return because vertex 5 has no outgoing edge. With no directed cycles, it is a __directed acyclic graph__. Next, we use vertex degrees and other measures to quantify graph structure.


### Graph Measures

The __degree__ $\deg(v_i)$ of a vertex $v_i\in\mathcal{V}$ counts the edges that touch it. In a directed graph, we distinguish:

* __In-degree__ $\deg^{\mathrm{in}}(v_i)$: the number of edges pointing into $v_i$.
* __Out-degree__ $\deg^{\mathrm{out}}(v_i)$: the number of edges pointing out of $v_i$.

The total degree is $\deg(v_i)=\deg^{\mathrm{in}}(v_i)+\deg^{\mathrm{out}}(v_i)$.

__How are vertex degrees related to the number of edges?__ Let $n=|\mathcal{V}|$ and $m=|\mathcal{E}|$. Summing the degrees in an undirected graph counts both endpoints of every edge. In a directed graph, each edge contributes once to an in-degree and once to an out-degree.

> __Handshaking identities:__
>
> $$
> \begin{aligned}
> \sum_{v_i\in\mathcal{V}}\deg(v_i)&=2m
> &&\text{(undirected)},\\
> \sum_{v_i\in\mathcal{V}}\deg^{\mathrm{in}}(v_i)
> &=\sum_{v_i\in\mathcal{V}}\deg^{\mathrm{out}}(v_i)=m
> &&\text{(directed)}.
> \end{aligned}
> $$

For an undirected graph with $n\geq1$, dividing the degree sum by the number of vertices gives the __average degree__:
$$
\bar d(\mathcal{G})
=\frac{1}{n}\sum_{v_i\in\mathcal{V}}\deg(v_i)
=\frac{2m}{n}.
$$
The average lies between the __minimum degree__ $\delta(\mathcal{G})$ and __maximum degree__ $\Delta(\mathcal{G})$:
$$
\delta(\mathcal{G})\leq\bar d(\mathcal{G})\leq\Delta(\mathcal{G}).
$$
An undirected graph is __regular__ of degree $r$ when every vertex has degree $r$. In that case, the minimum, average, and maximum degrees all equal $r$.

__How many of the possible edges are present?__ For an undirected simple graph, each of the $n$ vertices can connect to $n-1$ others. Dividing by two avoids counting each edge twice. For a directed simple graph, the two directions represent different edges, so no division is needed.

> __Graph density:__
>
> For $n\geq2$, the maximum edge count and density are
> $$
> |\mathcal{E}|_{\max}=
> \begin{cases}
> n(n-1)/2 & \text{undirected},\\
> n(n-1) & \text{directed},
> \end{cases}
> \qquad
> \rho(\mathcal{G})=\frac{m}{|\mathcal{E}|_{\max}}.
> $$
>
> Density measures the fraction of possible edges that are present. A graph with $\rho$ close to 1 is __dense__; one with $\rho$ close to 0 is __sparse__. When $n<2$, no loop-free edges are possible, so we use the convention $\rho(\mathcal{G})=0$.

The graphs in the figure each have six vertices and seven edges. Their densities differ: the undirected graph has density $7/15$, while the directed graph has density $7/30$, because twice as many directed edges are possible.

Vertex and edge counts do not fully describe a graph's structure. For an undirected graph, the following measures describe distances and the ways vertices can be grouped:

* __Diameter__ $\operatorname{diam}(\mathcal{G})$: for a connected graph, the largest shortest-path distance between two vertices, measured in edges.
* __Clique number__ $\omega(\mathcal{G})$: the size of the largest set of vertices in which every pair is joined. For example, this could describe the largest group of people who all know each other.
* __Chromatic number__ $\chi(\mathcal{G})$: the fewest colors needed so that no edge joins vertices of the same color. If vertices represent classes and edges connect classes with shared students, the colors represent time slots that avoid scheduling conflicts.
* __Independence number__ $\alpha(\mathcal{G})$: the size of the largest set of vertices with no edge between them. If edges represent conflicts between activities, an independent set describes activities that can run at the same time.

__Why do we need more than vertex and edge counts?__ Consider a path graph, whose vertices form a single chain, and a star graph, whose outer vertices connect to one central vertex. For $n\geq4$, both have $n$ vertices and $n-1$ edges, so they have the same average degree and density. Their arrangements differ: the path has maximum degree 2 and diameter $n-1$, while the star has maximum degree $n-1$ and diameter 2.

Computing these measures requires knowing which vertices are connected. Next, we compare ways to store those connections and examine how the choice affects memory use and access costs.

___


## How are Graphs Stored?

Let's compare three ways to store a graph: an edge list, an adjacency matrix, and an adjacency list. Each organizes connections differently, affecting how we find an edge or visit a vertex's neighbors.

An __edge list__ stores one record per edge, containing its endpoints and, when present, its weight. Text files often use this format, including our worked example. Without an index, finding an edge or collecting a vertex's neighbors may require scanning the entire list. Vertices with no incident edges do not appear in these records, so we must record them separately.

An __adjacency matrix__ $\mathbf{A}$ for a graph with $n=|\mathcal{V}|$ vertices has $n$ rows and $n$ columns. Its entry $a_{ij}$ describes the edge from $v_i$ to $v_j$:

* __Unweighted:__ $a_{ij}=1$ if the edge exists and $a_{ij}=0$ otherwise.
* __Weighted:__ $a_{ij}=w_{ij}$ if the edge exists, where $w_{ij}$ is its weight. Zero can represent a missing edge when all edge weights are nonzero.

For an undirected graph, the matrix is symmetric, $a_{ij}=a_{ji}$; for a directed graph, it need not be. If zero is a valid weight, we must distinguish a missing edge from an edge of weight zero.

An __adjacency list__ stores one neighbor collection for each vertex. In a directed graph, each collection contains the targets of that vertex's outgoing edges. An unweighted list may store target identifiers in a `Dict{Int64,Vector{Int64}}`. A weighted list can store `(target, weight)` pairs in a `Dict{Int64,Vector{Tuple{Int64,Float64}}}`. The dictionary key identifies the source vertex; a vertex with no outgoing edges has an empty collection.

__How does an adjacency list differ from an edge-weight map?__ A flat dictionary such as `Dict{Tuple{Int64,Int64},Float64}` maps each `(source, target)` pair to its weight. It supports expected constant-time lookup when both endpoints are known, but collecting one source's neighbors requires scanning the edge keys. Pairing this map with an unweighted adjacency list provides direct weight lookup and efficient neighbor iteration.

### Storage and Access Costs

__What does it cost to store and use a graph?__ Let $n=|\mathcal{V}|$, $m=|\mathcal{E}|$, and $d_u=\deg^{\mathrm{out}}(u)$. As in [the L3b lecture](../../week-03/L3b/CHEME-5800-L3b-Lecture-StacksAndQueues-Fall-2026.ipynb), $\Theta(\cdot)$ describes a tight asymptotic growth rate, while $O(\cdot)$ gives an upper bound.

We determine storage requirements by counting what each representation holds:

* An __edge list__ stores $m$ edge records, requiring $\Theta(m)$ storage. Storing the vertex set adds $\Theta(n)$, giving $\Theta(n+m)$ overall.
* An __adjacency matrix__ stores all $n^2$ entries, including missing edges, so it requires $\Theta(n^2)$ storage.
* An __adjacency list__ stores one collection per vertex and one neighbor record per directed edge, requiring $\Theta(n+m)$ storage. For an undirected graph, each edge appears in both endpoint lists. The resulting $n+2m$ records still give $\Theta(n+m)$ storage.

__How does the representation affect access time?__ An edge list may require scanning every edge to find a specified pair or collect a vertex's neighbors. A matrix provides direct access to $a_{ij}$, but listing neighbors requires scanning a full row. An adjacency list scans only the source vertex's neighbor collection.

The table compares these operations for a directed graph. Edge searches use worst-case scan lengths; dictionary access uses its expected constant-time cost.

<table style="border-collapse:collapse; width:100%; font-size:inherit;">
<thead>
<tr style="border-bottom:1px solid #999;">
<th style="text-align:left; padding:6px 8px;">Representation</th>
<th style="text-align:center; padding:6px 8px;">Storage</th>
<th style="text-align:center; padding:6px 8px;">Find edge (<i>u</i>, <i>v</i>)</th>
<th style="text-align:center; padding:6px 8px;">List neighbors of <i>u</i></th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align:left; padding:6px 8px;">Edge list with vertex set</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(<i>n</i> + <i>m</i>)</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(1 + <i>m</i>)</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(1 + <i>m</i>)</td>
</tr>
<tr>
<td style="text-align:left; padding:6px 8px;">Adjacency matrix</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(<i>n</i><sup>2</sup>)</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(1)</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(<i>n</i>)</td>
</tr>
<tr>
<td style="text-align:left; padding:6px 8px;">Adjacency list</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(<i>n</i> + <i>m</i>)</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(1 + <i>d</i><sub><i>u</i></sub>)</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(1 + <i>d</i><sub><i>u</i></sub>)</td>
</tr>
</tbody>
</table>

The adjacency-list edge search assumes a linear scan of the neighbor vector. The added 1 accounts for constant work even when a list is empty. Pairing an adjacency list with an edge-weight map gives expected $\Theta(1)$ lookup for a known vertex pair while preserving efficient neighbor iteration.

We choose a representation based on the operations our calculation needs. A matrix can suit dense graphs with repeated edge lookups, provided it fits in memory. An adjacency list suits sparse-graph traversal because it stores and visits only existing neighbors. An edge list suits calculations that process every edge in turn.

[The L4b traversal lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) uses an adjacency list because breadth-first and depth-first search examine each visited vertex's outgoing neighbors.


### Worked Example: One Graph in Three Forms

We now use the seven weighted edges in [`data/SimpleGraph.txt`](data/SimpleGraph.txt) to construct an adjacency list and an adjacency matrix for a six-vertex directed graph.

* [The `read_weighted_edges(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.read_weighted_edges-Tuple%7BAbstractString%7D) loads the edge records into `edge_records`. We then use [the `adjacency_list(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.adjacency_list-Tuple%7BAny%7D) to construct the outgoing-neighbor lists in `adjacency`, and [the `adjacency_matrix(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.adjacency_matrix-Tuple%7BAny%7D) to construct the weighted matrix and its vertex ordering in `matrix_representation`.

Our adjacency list stores only neighbor identifiers because the traversal algorithms in L4b need connectivity. The edge records and matrix retain the weights. A weighted adjacency list could retain them too by storing `(target, weight)` pairs.


In [ ]:
# Build three representations of the same directed, weighted graph -
# The let block keeps temporary names inside it and returns the three notebook values.
edge_records, adjacency, matrix_representation = let
    
    # Locate and read the edge list -
    edge_path = joinpath(CHEME5800_L4A_DATA, "SimpleGraph.txt") # start from the L4a data folder, not pwd()
    records = read_weighted_edges(edge_path)                    # Vector of (source, target, weight) NamedTuples

    # Convert the edge records into two graph data structures -
    list = adjacency_list(records)                               # Dict: vertex id => sorted outgoing-neighbor ids
    matrix = adjacency_matrix(records)                           # NamedTuple: weighted matrix plus row/column vertex ids

    # Return values from the local scope -
    records, list, matrix                                       # return the three representations from let
end; # hide the full output; display the stored representations separately

Let's inspect the outgoing-neighbor lists alongside the weighted matrix. The `vertex_order` field in the displayed result identifies the vertex associated with each matrix row and column. This ordering lets us interpret the matrix even when vertex identifiers are not consecutive integers.


In [ ]:
# Display the out-neighbor list beside the equivalent weighted matrix -
(
    adjacency = adjacency,                            # each key is a vertex; each value is its sorted out-neighbor vector
    vertex_order = matrix_representation.vertex_ids, # position i in this vector identifies matrix row and column i
    matrix = matrix_representation.matrix,            # entry (i,j) is the weight from vertex_order[i] to vertex_order[j]
)

Vertex 1 points to vertices 2 and 3, so `adjacency[1]` contains `[2, 3]`. The corresponding matrix entries contain the edge weights, 10 and 100. Vertex 6 has an empty neighbor vector and a zero matrix row because no edge leaves it.

[The `representation_report(...)` function](https://varnerlab.org/CHEME-5800-CourseRepository-Fall-2026/dev/library/#VLDataScienceMachineLearningPackage.representation_report-Tuple%7BAny%7D) summarizes our six-vertex graph: its vertex and edge counts, directed density, matrix entries, and adjacency-list items. We store this report in `representation`.


In [ ]:
# Summarize the six-vertex graph and count its storage entries -
representation = representation_report(edge_records)


For our six-vertex graph, the matrix reserves 36 entries, while the adjacency list contains 6 vertex keys and 7 neighbor slots.

__How does storage scale as the graph grows?__ Big Theta describes storage growth as well as running time. For $n$ vertices and $m$ edges, the three representations have the following storage requirements:

<table style="border-collapse:collapse; width:100%; font-size:inherit;">
<thead>
<tr style="border-bottom:1px solid #999;">
<th style="text-align:left; padding:6px 8px;">Representation</th>
<th style="text-align:center; padding:6px 8px;">Storage scaling</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align:left; padding:6px 8px;">Edge list with an explicit vertex set</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(<i>n</i> + <i>m</i>)</td>
</tr>
<tr>
<td style="text-align:left; padding:6px 8px;">Adjacency matrix</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(<i>n</i><sup>2</sup>)</td>
</tr>
<tr>
<td style="text-align:left; padding:6px 8px;">Adjacency list</td>
<td style="text-align:center; padding:6px 8px; white-space:nowrap;">Θ(<i>n</i> + <i>m</i>)</td>
</tr>
</tbody>
</table>

The edge records alone require $\Theta(m)$ storage. Adding one weight per edge changes the amount of memory, but not the asymptotic scaling.

__Example__: Suppose each vertex has $k=10$ outgoing edges, so $m=kn$. Since $k$ is fixed, the storage requirements become
$$
S_{\mathrm{list}}(n)=\Theta(n+kn)=\Theta(n),
\qquad
S_{\mathrm{matrix}}(n)=\Theta(n^2).
$$
Thus, doubling the number of vertices approximately doubles the list's storage but quadruples the matrix's storage. For a dense graph with $m=\Theta(n^2)$, both representations instead require $\Theta(n^2)$ storage.

The following tests check the six-vertex counts, density, adjacency list, one matrix weight, and storage-entry counts against the input data.


In [ ]:
# Check that every representation stores the expected graph -
@testset "graph representations" begin
    # Check the structure encoded by the input and both representations -
    @test representation.vertices == 6                       # the edge endpoints use the six vertex ids 1,...,6
    @test representation.edges == 7                          # the file contains seven directed source-to-target records
    @test representation.density ≈ 7 / 30                    # 7 observed edges divided by 6(6-1) possible loop-free edges
    @test adjacency[1] == [2, 3]                             # vertex 1 points outward to vertices 2 and 3, in sorted order
    @test matrix_representation.matrix[1, 2] == 10.0         # entry (1,2) stores the weight of edge 1 → 2

    # Check the exact storage counts used in the lecture comparison -
    @test representation.matrix_entries == 36                # a 6×6 matrix reserves one entry for every ordered pair
    @test representation.adjacency_list_entries == 13        # six dictionary keys plus one target entry per edge
end

___

## Graph Families

Let's examine three graph families: complete graphs, bipartite graphs, and trees. Their defining properties let us say more about connectivity, graph measures, and storage requirements.

### Complete Graphs

A __complete graph__ $K_n$ is a simple undirected graph on $n$ vertices with an edge between every pair of distinct vertices. Once $n$ is known, the graph is fixed except for the vertex names.

<div>
    <center>
        <img src="figs/Fig-Complete-Graph-Schematic.svg" width="900" alt="The complete graph K5 with five vertices and all ten possible edges. The four edges incident to vertex 1 are highlighted, showing that every vertex has degree 4."/>
    </center>
</div>

The figure shows $K_5$, with the four edges touching vertex 1 highlighted. Every vertex connects to the other four vertices.

__How do these counts generalize to $K_n$?__ Every vertex connects to $n-1$ others, so $K_n$ is regular of degree $n-1$. The handshaking identity gives
$$
2|\mathcal{E}|=\sum_{v_i\in\mathcal{V}}\deg(v_i)=n(n-1),
$$
and therefore $|\mathcal{E}|=n(n-1)/2$. Every possible edge is present, and every pair of distinct vertices is one edge apart.

> __Counts and measures for $K_n$:__
>
> For $n\geq2$,
> $$
> \begin{aligned}
> |\mathcal{E}|&=\binom{n}{2}=\frac{n(n-1)}{2},
> &\deg(v_i)&=n-1,\\
> \rho(K_n)&=1,
> &\operatorname{diam}(K_n)&=1.
> \end{aligned}
> $$

The full vertex set is a clique. Because every pair is adjacent, each vertex needs a different color, and an independent set can contain only one vertex. Thus,
$$
\omega(K_n)=\chi(K_n)=n,
\qquad
\alpha(K_n)=1.
$$

A complete graph has the largest possible edge count among simple undirected graphs with $n$ vertices. Since $|\mathcal{E}|=\Theta(n^2)$, both adjacency matrices and adjacency lists require $\Theta(n^2)$ storage for this family.

__What are some examples of complete graphs?__ A [round-robin schedule](https://nrich.maths.org/articles/tournament-scheduling) has the edge pattern of $K_n$: every team plays every other team. The [traveling-salesman problem](https://www.math.uwaterloo.ca/tsp/college/index.html) uses a weighted complete graph when every city pair has a symmetric travel cost. The goal is to find the least-cost tour that visits every city once and returns to the starting city.


### Bipartite Graphs

A graph $\mathcal{G}=(\mathcal{V},\mathcal{E})$ is __bipartite__ if its vertices can be divided into two disjoint groups, $\mathcal{V}_1$ and $\mathcal{V}_2$, with every edge joining a vertex in one group to a vertex in the other. For example, vertices might represent workers and jobs, with an edge indicating that a worker is qualified for a job.

__What makes a bipartite graph complete?__ A bipartite graph need not contain every possible edge between its groups. A __complete bipartite graph__ $K_{m,n}$ contains all of them. Here, $m=|\mathcal{V}_1|$ and $n=|\mathcal{V}_2|$ denote the group sizes.

<div>
    <center>
        <img src="figs/Fig-Bipartite-Graph-Schematic/Fig-Bipartite-Graph-Schematic.svg" width="900" alt="The complete bipartite graph K3,4 with three blue vertices in the left group and four gold vertices in the right group. All twelve edges cross between the groups, and three highlighted edges form a matching that covers the left group."/>
    </center>
</div>

The figure shows $K_{3,4}$: three vertices in one group and four in the other, with every possible connection between the groups.

Each vertex in $\mathcal{V}_1$ connects to all $n$ vertices in $\mathcal{V}_2$, and each vertex in $\mathcal{V}_2$ connects to all $m$ vertices in $\mathcal{V}_1$. Thus,
$$
|\mathcal{E}(K_{m,n})|=mn,
\qquad
\deg(v)=
\begin{cases}
n & v\in\mathcal{V}_1,\\
m & v\in\mathcal{V}_2.
\end{cases}
$$
For $m,n\geq1$, the graph is regular exactly when $m=n$.

__How can we recognize a bipartite graph?__ Give the vertices in $\mathcal{V}_1$ one color and those in $\mathcal{V}_2$ another. Every edge then joins different colors. Conversely, any valid coloring using at most two colors divides the vertices into groups with no edges within either group.

> __Three equivalent tests for a bipartite graph:__
>
> For an undirected graph $\mathcal{G}$, the following statements are equivalent:
>
> 1. $\mathcal{G}$ is bipartite.
> 2. $\mathcal{G}$ can be colored with at most two colors, so $\chi(\mathcal{G})\leq2$.
> 3. $\mathcal{G}$ has no cycle of odd length.

Alternating two colors around an odd cycle gives its first and last vertices the same color. Conversely, if no odd cycle exists, breadth-first search in each connected component groups vertices by even or odd distance from the start. An edge within either group would create an odd cycle, so these groups form a bipartition.

The coloring test can be carried out with a graph traversal:

1. Mark every vertex uncolored.
2. Choose an uncolored vertex, give it color 1, and run breadth-first or depth-first search through its connected component.
3. Give each uncolored neighbor the opposite color of the current vertex. If an edge joins two vertices with the same color, stop: the graph is not bipartite.
4. Repeat from another uncolored vertex until every connected component has been checked. If no conflict occurs, the two color classes are $\mathcal{V}_{1}$ and $\mathcal{V}_{2}$.

With an undirected adjacency list, this test visits each vertex once and reads each edge from both endpoint lists. Its running time is $\Theta(|\mathcal{V}|+|\mathcal{E}|)$. [The L4b lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb) develops the breadth-first and depth-first searches used by this test.

A __matching__ is a set of edges that do not share vertices. In our worker–job example, each selected edge assigns a worker to a job, with no worker or job used twice. A matching __covers__ $\mathcal{V}_1$ when every vertex in that group belongs to a selected edge.

The three highlighted edges in the figure form a matching that covers $\mathcal{V}_1$. One vertex in $\mathcal{V}_2$ remains unmatched.

__When can every worker receive a different job?__ Having enough jobs overall is necessary, but the available connections also matter. For example, three workers who are collectively qualified for only two jobs cannot all receive different assignments. Hall's theorem makes this requirement precise.

> __[Hall's theorem](https://ocw.mit.edu/courses/6-042j-mathematics-for-computer-science-spring-2015/mit6_042js15_textbook.pdf):__
>
> For a bipartite graph with groups $\mathcal{V}_1$ and $\mathcal{V}_2$, let $N(S)\subseteq\mathcal{V}_2$ contain all neighbors of the vertices in $S\subseteq\mathcal{V}_1$. A matching that covers $\mathcal{V}_1$ exists if and only if
> $$
> |N(S)|\geq|S|
> \qquad\text{for every }S\subseteq\mathcal{V}_1.
> $$

Every group of workers must therefore have at least as many available jobs as workers. The theorem tells us that this condition is sufficient as well as necessary.

A matching is __perfect__ when it covers both vertex groups. This requires $|\mathcal{V}_1|=|\mathcal{V}_2|$, together with Hall's condition. The highlighted matching in $K_{3,4}$ covers the smaller group but is not perfect.

__How does the bipartition appear in an adjacency matrix?__ List the $m$ vertices of $\mathcal{V}_1$ first, followed by the $n$ vertices of $\mathcal{V}_2$, using this order for both rows and columns. Edges must cross between the groups, but they may be directed or undirected.

> __Bipartite adjacency matrix:__
>
> For a directed bipartite graph,
> $$
> \mathbf{A}=
> \begin{bmatrix}
> \mathbf{0} & \mathbf{B}\\
> \mathbf{C} & \mathbf{0}
> \end{bmatrix},
> $$
> where $\mathbf{B}\in\mathbb{R}^{m\times n}$ records edges from $\mathcal{V}_1$ to $\mathcal{V}_2$, and $\mathbf{C}\in\mathbb{R}^{n\times m}$ records edges in the reverse direction. The diagonal blocks are zero because no edge joins vertices within the same group.

The two off-diagonal blocks can differ. If all edges point from $\mathcal{V}_1$ to $\mathcal{V}_2$, then $\mathbf{C}=\mathbf{0}$. For an undirected graph, symmetry gives $\mathbf{C}=\mathbf{B}^{\top}$, so
$$
\mathbf{A}=
\begin{bmatrix}
\mathbf{0} & \mathbf{B}\\
\mathbf{B}^{\top} & \mathbf{0}
\end{bmatrix}.
$$
Here, the $m\times n$ __biadjacency matrix__ $\mathbf{B}$ has rows corresponding to $\mathcal{V}_1$ and columns corresponding to $\mathcal{V}_2$. For an unweighted graph, its entries are 1 for connected vertex pairs and 0 otherwise. In the illustrated undirected $K_{3,4}$, $\mathbf{B}$ is a $3\times4$ matrix of ones.

For the undirected case, storing $\mathbf{B}$ and the vertex ordering is enough to reconstruct $\mathbf{A}$. A general directed bipartite graph requires both $\mathbf{B}$ and $\mathbf{C}$. An adjacency list remains useful when relatively few connections exist between the groups.

### Trees

A __tree__ $\mathcal{T}=(\mathcal{V},\mathcal{E})$ is a connected undirected graph with no cycles. Connectivity guarantees a path between every pair of vertices. If two distinct paths joined the same pair, their union would contain a cycle, so the path must be unique.

This unique-path property explains why removing any edge disconnects a tree. Conversely, adding an edge between previously nonadjacent vertices creates one cycle: the new edge closes the existing path between its endpoints.

> __Six equivalent characterizations of a tree:__
>
> For a simple undirected graph $\mathcal{G}$ with $n\geq1$ vertices, the following statements are equivalent:
>
> 1. $\mathcal{G}$ is a tree.
> 2. $\mathcal{G}$ is connected and has exactly $n-1$ edges.
> 3. $\mathcal{G}$ has no cycle and has exactly $n-1$ edges.
> 4. $\mathcal{G}$ is connected, and removing any edge disconnects it.
> 5. $\mathcal{G}$ has no cycle, and adding any missing edge creates exactly one cycle.
> 6. Any two vertices are joined by exactly one path.

A tree therefore has $n-1$ edges. For $n\geq2$, its density is
$$
\rho(\mathcal{T})
=\frac{n-1}{n(n-1)/2}
=\frac{2}{n}.
$$
The density tends to zero as $n$ grows, even though the tree remains connected. An adjacency list requires $\Theta(n)$ storage because the number of edges grows linearly with the number of vertices.

__How do we describe a hierarchy using a tree?__ Choose one vertex as the __root__. Every other vertex has one __parent__: the next vertex on its path to the root. Vertices with the same parent are its __children__. A vertex with no children is a __leaf__. The __depth__ of a vertex counts the edges from the root to that vertex, and the tree's __height__ is its largest vertex depth.

<div>
    <center>
        <img src="figs/Fig-General-Tree-Schematic/Fig-General-Tree-Schematic.svg" width="880" alt="A rooted tree with root r at depth 0, internal vertices at depths 1 and 2, leaves at depths 2 and 3, and height 3."/>
    </center>
</div>

Every connected undirected graph contains a __spanning tree__: a subgraph containing all its vertices and enough edges to keep them connected without cycles. A graph with no cycles is a __forest__; each connected component is a tree.

___


## Looking Ahead: Traversal and Shortest Paths

We have described graph structure and compared ways to store its connections. Next, we use these representations to explore graphs and find routes through them.

In [the L4b lab](../L4b/CHEME-5800-L4b-Lab-BreadthFirstAndDepthFirstSearch-Fall-2026.ipynb), we use the six-vertex directed graph to compare __breadth-first search__, which uses a queue to visit vertices in layers by their distance from the start, and __depth-first search__, which follows a branch before backtracking. Both algorithms use outgoing-neighbor lists; edge weights do not affect their traversal order.

In [the L4c lecture](../L4c/CHEME-5800-L4c-Lecture-ShortestPathAlgorithms-Fall-2026.ipynb), we use edge weights to find routes with the smallest total cost. Our Dijkstra implementation uses a weighted adjacency list to examine outgoing edges, while Bellman–Ford uses an edge list to revisit every edge during each pass. These algorithms illustrate how the operations we need determine which representation is useful.

___


## Summary

In this lecture, we developed the language of graphs, examined three graph families, and compared their storage representations. These ideas connect the structure of a problem with the data structures used to solve it.

> __Key Takeaways:__
>
> * __Graph structure and measures:__ We defined walks, paths, cycles, and connectivity to describe how vertices are connected. We used the handshaking identities to relate vertex degrees to edge counts and calculated density as the fraction of possible edges present. We also introduced measures of distance and vertex grouping to describe properties that vertex and edge counts alone cannot determine.
>
> * __Complete graphs, bipartite graphs, and trees:__ We examined the defining properties of three graph families and used them to derive edge counts and other measures. For bipartite graphs, we connected coloring with the absence of odd cycles and used Hall's theorem to characterize when a matching can cover one vertex group. For trees, we related connectivity and the absence of cycles to unique paths and the $n-1$ edge count.
>
> * __Storage and access costs:__ We represented a graph using edge records, an adjacency list, and an adjacency matrix. We compared their storage growth and the work required for edge lookup and neighbor iteration. This explained why adjacency lists suit sparse-graph traversal, while matrices provide direct edge access at the cost of reserving an entry for every vertex pair.
